<a href="https://colab.research.google.com/github/PranjalAsthana/InfoRetrievalMethods/blob/main/recallprecisionf1scoreonretrievedqueries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
def evaluate_query(retrieved_docs, relevant_docs):
    # Convert lists to sets to find overlapping items easily
    retrieved_set = set(retrieved_docs)
    relevant_set = set(relevant_docs)

    # True Positives: documents that are both retrieved AND relevant
    true_positives = len(retrieved_set.intersection(relevant_set))

    # Calculate Precision (avoid division by zero if nothing was retrieved)
    precision = true_positives / len(retrieved_set) if retrieved_set else 0.0

    # Calculate Recall (avoid division by zero if no relevant docs exist)
    recall = true_positives / len(relevant_set) if relevant_set else 0.0

    # Calculate F1-Score
    if (precision + recall) > 0:
        f1_score = 2 * (precision * recall) / (precision + recall)
    else:
        f1_score = 0.0

    return {
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1-Score": round(f1_score, 4)
    }

# --- Example Usage ---
# All documents in the database that actually match the user's intent
ground_truth_relevant = ["doc_1", "doc_3", "doc_5", "doc_8", "doc_12"]

# Documents returned by your search query
query_results = ["doc_1", "doc_2", "doc_3", "doc_9"]

metrics = evaluate_query(query_results, ground_truth_relevant)
print(metrics)
# Output: {'Precision': 0.5, 'Recall': 0.4, 'F1-Score': 0.4444}


{'Precision': 0.5, 'Recall': 0.4, 'F1-Score': 0.4444}


In [5]:
def calculate_ir_metrics(retrieved_docs, relevant_docs, k):
    """
    Calculates Precision@k, Recall@k, F1@k, and Average Precision (AP@k)
    for a ranked list of retrieved documents.
    """
    # Slice the retrieved list to the top k items
    top_k = retrieved_docs[:k]
    relevant_set = set(relevant_docs)

    # Count how many of the top k items are in the relevant set
    relevant_retrieved = sum(1 for doc in top_k if doc in relevant_set)

    # 1. Precision@k
    precision_at_k = relevant_retrieved / k if k > 0 else 0.0

    # 2. Recall@k
    total_relevant = len(relevant_set)
    recall_at_k = relevant_retrieved / total_relevant if total_relevant > 0 else 0.0

    # 3. F1@k (Harmonic Mean)
    if (precision_at_k + recall_at_k) > 0:
        f1_at_k = 2 * (precision_at_k * recall_at_k) / (precision_at_k + recall_at_k)
    else:
        f1_at_k = 0.0

    # 4. Average Precision (AP@k) - accounts for exact rank order
    precisions_at_hits = []
    hits = 0
    for rank, doc in enumerate(top_k, start=1):
        if doc in relevant_set:
            hits += 1
            precisions_at_hits.append(hits / rank)

    ap_at_k = sum(precisions_at_hits) / total_relevant if total_relevant > 0 else 0.0

    return {
        f"Precision@{k}": round(precision_at_k, 4),
        f"Recall@{k}": round(recall_at_k, 4),
        f"F1@{k}": round(f1_at_k, 4),
        f"AP@{k}": round(ap_at_k, 4)
    }

# --- Example Usage ---
# Ranked list returned by your search engine (Rank 1 is first)
ranked_results = ["doc_4", "doc_1", "doc_9", "doc_2", "doc_7", "doc_3", "doc_8", "doc_5"]

# Ground truth: All relevant documents in the database for this query
ground_truth = ["doc_1", "doc_2", "doc_3", "doc_5", "doc_10"]

# Evaluate at cutoff k = 5
metrics = calculate_ir_metrics(ranked_results, ground_truth, k=5)

for metric, score in metrics.items():
    print(f"{metric}: {score}")


Precision@5: 0.4
Recall@5: 0.4
F1@5: 0.4
AP@5: 0.2
